In [5]:
import os
import re
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageSequence
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# =========================
# Paths
# =========================
base_dir = r'N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\FSLeyes_outputs'
outdir   = r'N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\FSLeyes_outputs\merged_frames_12views'
cmap_file = r'N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\nih_new_iso.cmap'
os.makedirs(outdir, exist_ok=True)

# =========================
# File mapping
# =========================
# Rows: pleasant, neutral, unpleasant
# Cols: LO, RO, LI, RI
gif_files = {
    'pleasant': {
        'LO': os.path.join(base_dir, 'pleasant_LO.gif'),
        'RO': os.path.join(base_dir, 'pleasant_RO.gif'),
        'LI': os.path.join(base_dir, 'pleasant_LI.gif'),
        'RI': os.path.join(base_dir, 'pleasant_RI.gif'),
    },
    'neutral': {
        'LO': os.path.join(base_dir, 'neutral_LO.gif'),
        'RO': os.path.join(base_dir, 'neutral_RO.gif'),
        'LI': os.path.join(base_dir, 'neutral_LI.gif'),
        'RI': os.path.join(base_dir, 'neutral_RI.gif'),
    },
    'unpleasant': {
        'LO': os.path.join(base_dir, 'unpleasant_LO.gif'),
        'RO': os.path.join(base_dir, 'unpleasant_RO.gif'),
        'LI': os.path.join(base_dir, 'unpleasant_LI.gif'),
        'RI': os.path.join(base_dir, 'unpleasant_RI.gif'),
    }
}

row_order = ['pleasant', 'neutral', 'unpleasant']
col_order = ['LO', 'RO', 'LI', 'RI']
col_titles = {'LO': 'Left Out', 'RO': 'Right Out', 'LI': 'Left In', 'RI': 'Right In'}
row_titles = {'pleasant': 'Pleasant', 'neutral': 'Neutral', 'unpleasant': 'Unpleasant'}

# =========================
# Font setup
# =========================
font_path = r"C:\Windows\Fonts\arial.ttf"
try:
    font_tile = ImageFont.truetype(font_path, 34)
    font_col  = ImageFont.truetype(font_path, 40)
    font_row  = ImageFont.truetype(font_path, 40)
    font_time = ImageFont.truetype(font_path, 40)
    font_bar  = ImageFont.truetype(font_path, 26)
except:
    font_tile = ImageFont.load_default()
    font_col  = ImageFont.load_default()
    font_row  = ImageFont.load_default()
    font_time = ImageFont.load_default()
    font_bar  = ImageFont.load_default()

# =========================
# Helpers
# =========================
def draw_text_with_outline(draw, xy, text, font, fill='white', outline='black', width=2, anchor=None):
    x, y = xy
    for dx, dy in [(-width,0),(width,0),(0,-width),(0,width),
                   (-width,-width),(-width,width),(width,-width),(width,width)]:
        draw.text((x+dx, y+dy), text, font=font, fill=outline, anchor=anchor)
    draw.text((x, y), text, font=font, fill=fill, anchor=anchor)

def get_text_size(draw, text, font):
    try:
        bbox = draw.textbbox((0, 0), text, font=font)
        return bbox[2] - bbox[0], bbox[3] - bbox[1]
    except AttributeError:
        return draw.textsize(text, font=font)

def center_crop_or_pad(im, target_w, target_h, fill=(0, 0, 0, 0)):
    """Center-crop if larger; center-pad if smaller."""
    im = im.convert('RGBA')
    w, h = im.size

    # crop width
    if w > target_w:
        left = (w - target_w) // 2
        im = im.crop((left, 0, left + target_w, h))
        w = target_w

    # crop height
    if h > target_h:
        top = (h - target_h) // 2
        im = im.crop((0, top, w, top + target_h))
        h = target_h

    # pad if needed
    if w < target_w or h < target_h:
        canvas = Image.new('RGBA', (target_w, target_h), fill)
        x = (target_w - w) // 2
        y = (target_h - h) // 2
        canvas.paste(im, (x, y))
        im = canvas

    return im

def load_custom_cmap(cmap_path):
    """
    Load a custom colormap from .camp/.clut-like text file.
    Supports rows like:
    r g b
    or
    idx r g b
    Values can be 0-255 or 0-1.
    """
    rows = []
    with open(cmap_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith('#') or line.startswith('//') or line.startswith('%'):
                continue

            parts = re.split(r'[\s,;]+', line)
            nums = []
            for p in parts:
                try:
                    nums.append(float(p))
                except:
                    pass

            if len(nums) >= 3:
                if len(nums) >= 4:
                    nums = nums[-3:]   # keep RGB if first column is index
                else:
                    nums = nums[:3]

                rows.append(nums)

    if len(rows) == 0:
        raise ValueError(f"Could not parse colormap file: {cmap_path}")

    arr = np.array(rows, dtype=float)

    # normalize if 0-255
    if arr.max() > 1.0:
        arr = arr / 255.0

    arr = np.clip(arr, 0, 1)
    return ListedColormap(arr, name='custom_nih')

def make_colorbar(width=500, bar_height=18, text_height=38,
                  vmin=-0.12, vmax=0.12, cmap=None):
    total_height = bar_height + text_height
    fig, ax = plt.subplots(figsize=(width / 100, total_height / 100), dpi=100)
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

    grad = np.linspace(vmin, vmax, width)
    grad = np.tile(grad, (bar_height, 1))

    ax.imshow(
        grad,
        aspect='auto',
        cmap=cmap,
        origin='lower',
        extent=[0, width, text_height, text_height + bar_height]
    )
    ax.axis('off')
    fig.canvas.draw()
    img = np.array(fig.canvas.renderer.buffer_rgba())[..., :3]
    plt.close(fig)

    pil_img = Image.fromarray(img)
    draw = ImageDraw.Draw(pil_img)

    y = text_height - 28
    labels = [(0, f'{vmin:.2f}', 'left'),
              (width // 2, '0', 'center'),
              (width, f'{vmax:.2f}', 'right')]

    for x, txt, align in labels:
        tw, th = get_text_size(draw, txt, font_bar)
        if align == 'left':
            tx = 0
        elif align == 'center':
            tx = x - tw // 2
        else:
            tx = x - tw
        draw_text_with_outline(draw, (tx, y), txt, font_bar, fill='white', outline='black', width=1)

    return pil_img

# =========================
# Open GIFs and inspect sizes
# =========================
gif_objs = {}
widths = []
frame_counts = []

for emo in row_order:
    gif_objs[emo] = {}
    for view in col_order:
        path = gif_files[emo][view]
        if not os.path.exists(path):
            raise FileNotFoundError(path)

        gif = Image.open(path)
        gif_objs[emo][view] = gif
        widths.append(gif.size[0])
        frame_counts.append(getattr(gif, "n_frames", sum(1 for _ in ImageSequence.Iterator(gif))))

common_w = min(widths)
target_h = 600   # centered crop height
n_frames = min(frame_counts)

print(f"Common width: {common_w}")
print(f"Target height: {target_h}")
print(f"Using {n_frames} frames")

# =========================
# Layout
# =========================
left_margin = 260
right_margin = 40
top_margin = 150
bottom_margin = 40

gap_x = 25
gap_y = 35
extra_gap_between_out_in = 100  # between RO and LI

# x positions for 4 columns
x_positions = []
x = left_margin
for j, col in enumerate(col_order):
    x_positions.append(x)
    x += common_w
    if j == 0:
        x += gap_x
    elif j == 1:
        x += extra_gap_between_out_in
    elif j == 2:
        x += gap_x

canvas_w = x + right_margin
canvas_h = top_margin + 3 * target_h + 2 * gap_y + bottom_margin

# custom colormap
custom_cmap = load_custom_cmap(cmap_file)

# =========================
# Main loop
# =========================
for i in range(n_frames):
    canvas = Image.new('RGBA', (canvas_w, canvas_h), (0, 0, 0, 0))
    draw = ImageDraw.Draw(canvas)

    # ---- Column titles ----
    for j, col in enumerate(col_order):
        col_title = col_titles[col]
        cx = x_positions[j] + common_w // 2
        cy = top_margin - 55
        draw_text_with_outline(draw, (cx, cy), col_title, font_col, fill='white', outline='black', width=2, anchor='mm')

    # ---- Row titles + tiles ----
    for r, emo in enumerate(row_order):
        y0 = top_margin + r * (target_h + gap_y)

        # row label on left
        ry = y0 + target_h // 2
        draw_text_with_outline(draw, (left_margin - 40, ry), row_titles[emo], font_row,
                               fill='white', outline='black', width=2, anchor='rm')

        for c, view in enumerate(col_order):
            gif = gif_objs[emo][view]
            gif.seek(i)
            frame = gif.convert('RGBA')

            # crop/pad to common width + 600 height
            frame = center_crop_or_pad(frame, common_w, target_h, fill=(0, 0, 0, 0))

            # optional small tile label at bottom
            tile_draw = ImageDraw.Draw(frame)
            short_label = f"{emo[:1].upper()}_{view}"
            tw, th = get_text_size(tile_draw, short_label, font_tile)
            draw_text_with_outline(tile_draw,
                                   ((common_w - tw) // 2, target_h - th - 12),
                                   short_label, font_tile, fill='white', outline='black', width=2)

            canvas.paste(frame, (x_positions[c], y0))

    # ---- Colorbar centered at top ----
    colorbar_width = 520
    colorbar = make_colorbar(
        width=colorbar_width,
        bar_height=18,
        text_height=40,
        vmin=-0.12,
        vmax=0.12,
        cmap=custom_cmap
    )
    cb_x = (canvas_w - colorbar_width) // 2
    cb_y = 18
    canvas.paste(colorbar, (cb_x, cb_y))

    # ---- Timestamp top-right ----
    # last frame = 2000 ms, each earlier frame -4 ms
    last_idx = n_frames - 1
    ms = 2000 - 4 * (last_idx - i)
    ts = f"{ms}ms"

    tw, th = get_text_size(draw, ts, font_time)
    tx = canvas_w - tw - 25
    ty = 20
    draw_text_with_outline(draw, (tx, ty), ts, font_time, fill='white', outline='black', width=2)

    # ---- Save ----
    outpath = os.path.join(outdir, f'merged_{i:03d}.png')
    canvas.save(outpath)

    if i % 50 == 0 or i == n_frames - 1:
        print(f"Saved {i+1}/{n_frames}: {outpath}")

print(f"\nAll merged frames saved in:\n{outdir}")


Common width: 688
Target height: 600
Using 574 frames
Saved 1/574: N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\FSLeyes_outputs\merged_frames_12views\merged_000.png
Saved 51/574: N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\FSLeyes_outputs\merged_frames_12views\merged_050.png
Saved 101/574: N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\FSLeyes_outputs\merged_frames_12views\merged_100.png
Saved 151/574: N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\FSLeyes_outputs\merged_frames_12views\merged_150.png
Saved 201/574: N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\FSLeyes_outputs\merged_frames_12views\merged_200.png
Saved 251/574: N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\FSLeyes_outputs\merged_frames_12views\merged_250.png
Saved 301/574: N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_EEG_fusion\FSLeyes_outputs\merged_frames_12views\merged_300.png
Saved 351/574: N:\Experimental_Data\yujunchen\

in WSL

In [ ]:
ffmpeg -framerate 4 \
  -i "/mnt/n/Experimental_Data/yujunchen/projects/IAPS_EEG_RSA/frames/merged_frames_inflated/merged_%03d.png" \
  -c:v libx264 -pix_fmt yuv420p -r 4 \
  "/mnt/n/Experimental_Data/yujunchen/projects/IAPS_EEG_RSA/merged_movie_inflated.mp4"
